In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import current_timestamp 
from pyspark.sql.functions import input_file_name
from pyspark.sql.functions import col

In [0]:
raw_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
dir_name = "/".join(raw_path.split("/")[:-2])
config_file_path = f"/Workspace{dir_name}/process_config.json"
# dbutils.widgets.removeAll()
dbutils.widgets.text("config_file_path", config_file_path)
temp = dbutils.widgets.get("config_file_path")
# print(temp)

In [0]:
with  open(dbutils.widgets.get("config_file_path"), "r") as file:
# with  open("/Workspace/Repos/vishnuayithem@gmail.com/zapflow/process_config.json", "r") as file:
  config = file.read()
config = json.loads(config)
# print(config)
# print(type(config))


In [0]:
#Schema for customers csv files
customers_schema = StructType([
    StructField("customer_id",StringType(),False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("pincode", IntegerType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("acquisition_channel", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("is_active", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
    

])

#Schema for dark_stores csv files

dark_stores_schema = StructType([
    StructField("store_id", StringType(), False),
    StructField("store_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("address", StringType(), True),
    StructField("pincode", IntegerType(), True),
    StructField("capacity_sqft", IntegerType(), True),
    StructField("max_orders_per_day", IntegerType(), True),
    StructField("operational_since", DateType(), True),
    StructField("store_status", StringType(), True),
    StructField("delivery_radius_km", DoubleType(), True),
    StructField("_data_quality_issue", StringType(), True)
])

#Schema for deliveries csv files

deliveries_schema = StructType([
    StructField("delivery_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("delivery_partner_id", StringType(), False),
    StructField("store_id", StringType(), False),
    StructField("pickup_timestamp", StringType(), True),
    StructField("delivered_timestamp", StringType(), True),
    StructField("delivery_minutes", DoubleType(), True),
    StructField("is_sla_breached", StringType(), True),
    StructField("delivery_status", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
])

#Schema for inventory csv files

inventory_schema = StructType([
    StructField("inventory_id", StringType(), False),
    StructField("store_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("snapshot_date", DateType(), True),
    StructField("units_available", IntegerType(), True),
    StructField("units_sold", IntegerType(), True),
    StructField("reorder_level", IntegerType(), True),
    StructField("is_out_of_stock", StringType(), True),
    StructField("is_weekend", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
])

#Schema for orders csv files

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("store_id", StringType(), False),
    StructField("order_timestamp", StringType(), True),
    StructField("order_items", StringType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("delivery_charge", DoubleType(), True),
    StructField("final_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("is_weekend", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
])

In [0]:
# Reading the Customers CSV file

customers_df = spark.read.format("csv").option("header", "true").schema(customer_schema).load(config["source_files"]["customers"])
customers_df = customers_df.withColumn("ingestion_time",current_timestamp()).withColumn("source_file", col("_metadata.file_path"))

# input_file_name() doesnt work, its showing not work for unity catalog enabled
# customers_df.show(truncate=False)
# customers_df.display()

# Reading the Dark Stores CSV file

dark_stores_df = spark.read.format("csv").option("header", "true").schema(dark_stores_schema).load(config["source_files"]["dark_stores"])
dark_stores_df = dark_stores_df.withColumn("ingestion_time",current_timestamp()).withColumn("source_file", col("_metadata.file_path"))
# dark_stores_df.show(truncate=False)
# dark_stores_df.display()

# Reading the Deliveries CSV file

deliveries_df = spark.read.format("csv").option("header", "true").schema(deliveries_schema).load(config["source_files"]["deliveries"])
deliveries_df = deliveries_df.withColumn("ingestion_time",current_timestamp()).withColumn("source_file", col("_metadata.file_path"))
# deliveries_df.show(truncate=False)
# deliveries_df.display()

# Reading the Inventory CSV file

inventory_df = spark.read.format("csv").option("header", "true").schema(inventory_schema) .load(config["source_files"]["inventory"])
inventory_df = inventory_df.withColumn("ingestion_time",current_timestamp()).withColumn("source_file", col("_metadata.file_path"))
# inventory_df.show(truncate=False)
# inventory_df.display()

# Reading the Orders CSV file

orders_df = spark.read.format("csv").option("header", "true").schema(orders_schema).load(config["source_files"]["orders"])
orders_df = orders_df.withColumn("ingestion_time",current_timestamp()).withColumn("source_file", col("_metadata.file_path"))
# orders_df.show(truncate=False)
# orders_df.display() 


In [0]:
# Customers table

customers_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.customers")

# Dark Stores table

dark_stores_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.dark_stores")

# Deliveries table

deliveries_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.deliveries")

# Inventory table

inventory_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.inventory")

# Orders table

orders_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.orders")